# ML-04 — Search Intelligence Data Contract

Work the sections **in order**. Simple words, honest numbers.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane:** Refresh / Content Opportunity Scoring

1. **One row means:** one webpage (`content_hash_id`) for one client (`client_hash_id`), built from daily fact rows.
2. **Tables:** `fact_content_daily_performance`, `dim_content`, `dim_clients`.
3. **Time window:** features from March 2026 (`2026-03-01` to `2026-03-31`); label from April 2026 (`2026-04-01` to `2026-04-30`). Decision point = `2026-03-31`.
4. **Label / proxy:** `is_declining` = 1 if April impressions drop more than 20% vs March (`future_impressions < 0.8 × prior_impressions`), for pages with `prior_impressions >= 100`.
5. **Deliberately excluded:** `trend_direction` and `trend_pct` (they can overlap the outcome window and leak the label).

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import getpass
from pathlib import Path

import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download
from IPython.display import display

pd.set_option("display.max_columns", None)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except ImportError:
        pass
if not HF_TOKEN:
    for root in [Path.cwd(), *Path.cwd().parents]:
        env = root / ".env"
        if not env.exists():
            continue
        for line in env.read_text().splitlines():
            line = line.strip()
            if line.startswith("HF_TOKEN="):
                HF_TOKEN = line.split("=", 1)[1].strip()
            elif line.startswith("hf_"):
                HF_TOKEN = line
        if HF_TOKEN:
            break
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

REPO = "FlyRank/internship-warehouse"

def pq(filename):
    path = hf_hub_download(
        repo_id=REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    return "read_parquet('" + path.replace("\\", "/") + "')"

con = duckdb.connect()
TABLES = {
    "dim_content": pq("dim_content.parquet"),
    "fact_daily_march": pq("fact_content_daily_performance/month=2026-03/data_0.parquet"),
    "fact_daily_april": pq("fact_content_daily_performance/month=2026-04/data_0.parquet"),
}

print("March rows:", con.sql(f"SELECT COUNT(*) FROM {TABLES['fact_daily_march']}").fetchone()[0])

March rows: 9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** `prior_impressions`, `prior_clicks`, `prior_avg_position`, `content_age_days`, `word_count`

**Label:** `is_declining`

**Context:** `client_hash_id`, `content_hash_id`

**Excluded:** `trend_direction`, `trend_pct` — overlap with the outcome window

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
MARCH = TABLES["fact_daily_march"]

print("FACT 1 — Grain (expect 0 rows):")
display(con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MARCH}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df())

print("FACT 2 — Row count and date span:")
display(con.sql(f"""
    SELECT COUNT(*) AS total_daily_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {MARCH}
""").df())

print("FACT 3 — GA4 availability (IS TRUE):")
display(con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
           AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) AS ga4_available_ratio
    FROM {MARCH}
""").df())

FACT 1 — Grain (expect 0 rows):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


FACT 2 — Row count and date span:


,total_daily_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


FACT 3 — GA4 availability (IS TRUE):


,total_rows,ga4_available_rows,ga4_available_ratio
0,9841378,413966,0.042064


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

march = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS prior_impressions,
           SUM(gsc_clicks) AS prior_clicks,
           AVG(gsc_avg_position) AS prior_avg_position
    FROM {TABLES['fact_daily_march']}
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
""").df()

april = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS future_impressions
    FROM {TABLES['fact_daily_april']}
    GROUP BY 1, 2
""").df()

meta = con.sql(f"""
    SELECT content_hash_id, word_count, content_created_date
    FROM {TABLES['dim_content']}
""").df()

df_features = march.merge(april, on=["client_hash_id", "content_hash_id"], how="left")
df_features = df_features.merge(meta, on="content_hash_id", how="left")
df_features["future_impressions"] = df_features["future_impressions"].fillna(0)
df_features["word_count"] = df_features["word_count"].fillna(df_features["word_count"].median())
df_features["content_age_days"] = (
    pd.Timestamp("2026-03-31") - pd.to_datetime(df_features["content_created_date"])
).dt.days.fillna(365)
df_features["is_declining"] = (
    df_features["future_impressions"] < 0.8 * df_features["prior_impressions"]
).astype(int)

honest_cols = [
    "prior_impressions", "prior_clicks", "prior_avg_position",
    "content_age_days", "word_count",
]

print(f"Feature frame shape: {df_features.shape}")
display(df_features[honest_cols + ["is_declining"]].head())

X = df_features[honest_cols]
y = df_features["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model_h = RandomForestClassifier(n_estimators=100, random_state=42)
model_h.fit(X_tr, y_tr)
auc_honest = roc_auc_score(y_te, model_h.predict_proba(X_te)[:, 1])
print(f"Honest ROC AUC: {auc_honest:.4f}")

df_features["leaked_future_impressions"] = df_features["future_impressions"]
X_leak = df_features[honest_cols + ["leaked_future_impressions"]]
X_tr_l, X_te_l, _, _ = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)

model_l = RandomForestClassifier(n_estimators=100, random_state=42)
model_l.fit(X_tr_l, y_tr)
auc_leaked = roc_auc_score(y_te, model_l.predict_proba(X_te_l)[:, 1])
print(f"Leaked ROC AUC: {auc_leaked:.4f}")

df_features = df_features.drop(columns=["leaked_future_impressions", "future_impressions", "content_created_date"])
print("Leaked column deleted.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (101441, 10)


,prior_impressions,prior_clicks,prior_avg_position,content_age_days,word_count,is_declining
0,127.0,1.0,6.596515,210,919,1
1,2751.0,0.0,48.800095,375,2773,0
2,2842.0,2.0,12.524805,375,2509,0
3,415.0,0.0,53.519219,375,2844,0
4,1764.0,2.0,60.133938,375,2566,0


Honest ROC AUC: 0.7489
Leaked ROC AUC: 0.9993
Leaked column deleted.


### Feature availability + leakage

- `prior_impressions` — **knowable at the decision moment because** it sums March GSC impressions only.
- `prior_clicks` — **knowable at the decision moment because** it sums March GSC clicks only.
- `prior_avg_position` — **knowable at the decision moment because** it uses March ranking data only.
- `content_age_days` — **knowable at the decision moment because** it uses the page creation date, which is in the past.
- `word_count` — **knowable at the decision moment because** it is static content metadata.

Adding `leaked_future_impressions` (April data used in the label) pushes ROC AUC toward ~1.0. That is leakage. The leaked column is deleted above.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation — unbalanced client history:** clients start tracking on different dates, so a fixed March window can treat "not tracked yet" as "zero traffic." Results are directional decision-support, not a full census of every client.

## Self-check

- [ ] Every section filled — markdown and code outputs visible
- [ ] Notebook runs top to bottom with no errors
- [ ] No client names, URLs, or tokens in the notebook
- [ ] Committed under `work/notebooks/` — submit repo URL on the card